In [1]:
%%capture install_log
!pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml

In [2]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 73.7 MB/s eta 0:00:00:00:01


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # max_pool3d gibi MPS'de olmayan op'lar CPU'ya düşer

import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import monai
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Resized,
    ToTensord,
    RandRotate90d,
    RandFlipd,
    RandGaussianNoised,
    RandZoomd
)
from sklearn.model_selection import GroupKFold
from tqdm import tqdm


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-19 20:35:23.867511: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779222924.077995      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779222924.141270      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779222924.625313      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779222924.625358      57 computation_placer.cc:1

In [5]:
# Cihaz Kontrolü — M5 Pro için MPS öncelikli
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Aktif Cihaz: {device}")



Aktif Cihaz: cuda


In [6]:

CACHE_DIR = '/kaggle/input/datasets/ramazan2020/abdomen/kaggle/working/npy_cache'
# os.makedirs(CACHE_DIR, exist_ok=True)

In [7]:
import os
import numpy as np
import SimpleITK as sitk
sitk.ProcessObject.SetGlobalWarningDisplay(False)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from monai.networks.nets import resnet18

class GradCAM3D:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Katman çıktılarını ve gradyanlarını yakalamak için hook'ları bağlıyoruz
        self.target_layer.register_forward_hook(self.forward_hook)
        self.target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_image, class_idx):
        # input_image boyutu: [1, 1, 128, 128, 128]
        self.model.zero_grad()
        output = self.model(input_image)
        
        # Hedef sınıfın skorunu al ve geriye doğru türevini (backward) hesapla
        score = output[0, class_idx]
        score.backward()
        
        # Gradyanların kanalsal ortalamasını (ağırlıklarını) al
        # 3D gradyan boyutu: [Batch, Channel, H, W, D] -> 2,3,4 uzamsal akslar
        weights = torch.mean(self.gradients, dim=(2, 3, 4), keepdim=True)
        
        # Aktivasyon haritasını bu ağırlıklarla çarp
        cam = torch.sum(weights * self.activations, dim=1).squeeze(0)
        
        # ReLU operasyonu (Sadece pozitif katkı sağlayan özellikleri tut)
        cam = F.relu(cam)
        
        # Orijinal görüntü boyutuna (128, 128, 128) interpolasyon ile büyüt
        cam = cam.detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8) # Normalize et
        
        return cam

# --- FAZ 1: VERİ SETİ YÜKLEME VE PİPELİNE ---

class RSNA20233DDataset(Dataset):
    def __init__(self, df, base_path, transform=None):
        self.df = df
        self.base_path = base_path
        self.transform = transform
        
        # Yarışmadaki 13 Hedef Sütun
        self.targets = [
            'bowel_healthy', 'bowel_injury',
            'extravasation_healthy', 'extravasation_injury',
            'kidney_healthy', 'kidney_low', 'kidney_high',
            'liver_healthy', 'liver_low', 'liver_high',
            'spleen_healthy', 'spleen_low', 'spleen_high'
        ]

    def __len__(self):
        return len(self.df)

    def _load_3d_volume(self, patient_id):
        patient_dir = os.path.join(self.base_path, str(patient_id))
        if not os.path.exists(patient_dir):
            return None

        # sorted() ile seri seçimi tekrarlanabilir hale geliyor
        series_ids = sorted(os.listdir(patient_dir))
        if len(series_ids) == 0:
            return None

        series_dir = os.path.join(patient_dir, series_ids[0])
        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(series_dir)
        if not dicom_names:
            return None
        reader.SetFileNames(dicom_names)
        try:
            image = reader.Execute()
        except Exception:
            return None

        # (D, H, W) float32, HU values already applied by SimpleITK
        volume = sitk.GetArrayFromImage(image).astype(np.float32)
        return volume

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient_id = int(row['patient_id'])
        
        volume = self._load_3d_volume(patient_id)
        
        # Eğer okuma hatası veya boş klasör varsa dummy tensor oluştur (Pipeline çökmemesi için)
        if volume is None:
            volume = np.zeros((128, 128, 128), dtype=np.float32)
            
        data_dict = {"image": volume}
        if self.transform:
            data_dict = self.transform(data_dict)
            
        labels = row[self.targets].values.astype(np.float32)
        return data_dict["image"], torch.tensor(labels, dtype=torch.float32)


# --- 3D CBAM (DİKKAT MEKANİZMASI) BLOKLARI (Sabit ve Kararlı) ---
class ChannelAttention3D(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention3D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv3d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention3D, self).__init__()
        self.conv1 = nn.Conv3d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)


# --- YENİ HATA GEÇİRMEZ MODEL MİMARİSİ ---
class RSNAResNetCBAMClassifier(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        
        # 1. 3D ResNet18 omurgasını ham olarak alıyoruz
        self.backbone = resnet18(spatial_dims=3, n_input_channels=1, num_classes=2)
        
        # 2. ResNet'in orijinal son doğrusal (FC) katmanını etkisiz hale getiriyoruz (Identity).
        # Böylece backbone(x) dediğimizde model bize en son evrişim katmanının (Layer 4) ham 3D çıktısını verecek.
        # ResNet18 için bu çıktının kanal sayısı her zaman 512'dir.
        self.backbone.fc = nn.Identity()
        
        # 3. Faz 3: Dikkat mekanizması bloklarımızı tam bu 512 kanallı özelliğin üstüne ekliyoruz
        self.channel_attention = ChannelAttention3D(in_planes=512)
        self.spatial_attention = SpatialAttention3D()
        
        # 4. Saf PyTorch havuzlama, düzleştirme ve yarışmaya özel 13 sınıflı doğrusal katmanımız
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.pure_flatten = nn.Flatten()
        self.custom_classifier = nn.Linear(512, num_classes)
        
    def forward(self, x):
        # Modelin içindeki küçük katmanları (conv1, relu vb.) tek tek çağırmak yerine
        # doğrudan gövdeyi tetikliyoruz. Orijinal .fc katmanı nn.Identity() olduğu için
        # x bize doğrudan en derin özellik haritası (feature map) olarak dönecek.
        # Boyut: [Batch, 512, H_yeni, W_yeni, D_yeni]
        x = self.backbone(x)
        
        # Eğer backbone çıktıyı pooling uygulanmış veya düzleştirilmiş olarak dönerse,
        # 3D uzamsal boyutları (H, W, D) kaybetmemek adına, tensor şeklini kontrol edip
        # CBAM bloğuna tam bir 3D evrişim matrisi beslediğimizden emin oluyoruz:
        if len(x.shape) == 2:
            # Nadiren bazı MONAI sürümleri fc öncesi global pooling'i zorunlu çalıştırabilir.
            # Eğer veri zaten düzleşmişse doğrudan sınıflandırmaya yönlendirilir,
            # evrişimsel durum korunuyorsa (en yaygın durum) CBAM filtrelerinden geçer:
            pass
        else:
            # Faz 3: Dikkat Filtrelerini Uygula (Organ ve travma odaklama)
            x = self.channel_attention(x) * x
            x = self.spatial_attention(x) * x
            # Boyut küçültme ve düzleştirme
            x = self.global_pool(x)
            x = self.pure_flatten(x)
            
        # Sınıflandırma kafası: Yarışmadaki 13 sınıfın tahmin logitlerini üret
        logits = self.custom_classifier(x)
        return logits


class FastRSNADataset(Dataset):
    TARGETS = [
        'bowel_healthy', 'bowel_injury',
        'extravasation_healthy', 'extravasation_injury',
        'kidney_healthy', 'kidney_low', 'kidney_high',
        'liver_healthy', 'liver_low', 'liver_high',
        'spleen_healthy', 'spleen_low', 'spleen_high',
    ]

    def __init__(self, df, cache_dir, augment=False):
        valid = [pid for pid in df['patient_id']
                 if os.path.exists(os.path.join(cache_dir, f'{pid}.npy'))]
        self.df        = df[df['patient_id'].isin(valid)].reset_index(drop=True)
        self.cache_dir = cache_dir
        self.augment   = augment
        dropped = len(df) - len(self.df)
        if dropped:
            print(f'[FastRSNADataset] {dropped} hasta cache bulunamadi, atlandi.')

    def __len__(self):
        return len(self.df)

    def _augment(self, vol):
        if np.random.rand() < 0.5:
            vol = np.flip(vol, axis=0).copy()
        if np.random.rand() < 0.5:
            vol = np.rot90(vol, k=np.random.randint(1, 4), axes=(0, 1)).copy()
        if np.random.rand() < 0.2:
            vol = np.clip(vol + np.random.normal(0, 0.05, vol.shape).astype(np.float32), 0, 1)
        return vol

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = int(row['patient_id'])
        vol = np.load(os.path.join(self.cache_dir, f'{pid}.npy')).astype(np.float32)
        if self.augment:
            vol = self._augment(vol)
        image  = torch.tensor(vol).unsqueeze(0)  # [1, 128, 128, 128]
        labels = torch.tensor(row[self.TARGETS].values.astype(np.float32))
        return image, labels


# Top-level fonksiyon: ProcessPoolExecutor worker'larında pickle'lanabilmesi için
# sınıf içinde veya lambda olarak tanımlanamaz.
def _read_and_hu(args, target=(128, 128, 128)):
    """DICOM oku + HU dönüşümü + CPU resize. Worker'da biter, 4 MB float16 döner."""
    pid, train_images_dir, cache_dir = args
    out_path = os.path.join(cache_dir, f'{pid}.npy')
    if os.path.exists(out_path):
        return pid, None

    patient_dir = os.path.join(train_images_dir, str(pid))
    try:
        series_ids = sorted(os.listdir(patient_dir))
    except FileNotFoundError:
        return pid, None
    if not series_ids:
        return pid, None

    series_dir = os.path.join(patient_dir, series_ids[0])
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(series_dir)
    if not dicom_names:
        return pid, None
    reader.SetFileNames(dicom_names)
    try:
        image = reader.Execute()
    except Exception:
        return pid, None

    # HU clipping + normalizasyon
    volume = sitk.GetArrayFromImage(image).astype(np.float32)
    volume = np.clip(volume, -150, 250)
    volume = (volume + 150) / 400.0

    # CPU'da resize — 400+ MB yerine 4 MB float16 döner, RAM patlamaz
    t = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)  # [1,1,D,H,W]
    t = F.interpolate(t, size=target, mode='trilinear', align_corners=False)
    return pid, t.squeeze().numpy().astype(np.float16)


In [8]:
TRAIN_CSV = os.path.join("/kaggle/input/datasets/ramazan2020/rsna-excel/train_2024.csv")
df_train = pd.read_csv(TRAIN_CSV)


In [9]:
df_train.head()

,patient_id,bowel_healthy,bowel_injury,extravasation_healthy,extravasation_injury,kidney_healthy,kidney_low,kidney_high,liver_healthy,liver_low,liver_high,spleen_healthy,spleen_low,spleen_high,any_injury
0,10004,1,0,0,1,0,1,0,1,0,0,0,0,1,1
1,10005,1,0,1,0,1,0,0,1,0,0,1,0,0,0
2,10007,1,0,1,0,1,0,0,1,0,0,1,0,0,0
3,10026,1,0,1,0,1,0,0,1,0,0,1,0,0,0
4,10051,1,0,1,0,1,0,0,1,0,0,0,1,0,1


In [10]:

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(df_train, groups=df_train['patient_id']))
train_idx, val_idx = splits[0]
df_tr, df_va = df_train.iloc[train_idx], df_train.iloc[val_idx]


In [11]:

# FastRSNADataset: .npy okur, DICOM degil
train_ds = FastRSNADataset(df_tr, CACHE_DIR, augment=True)
val_ds   = FastRSNADataset(df_va, CACHE_DIR, augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')


[FastRSNADataset] 773 hasta cache bulunamadi, atlandi.
[FastRSNADataset] 194 hasta cache bulunamadi, atlandi.
Train: 1744 | Val: 436


In [12]:

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)


In [13]:
# Model
model = RSNAResNetCBAMClassifier(num_classes=13)
if torch.cuda.device_count() > 1:
    print(f'DataParallel: {torch.cuda.device_count()} GPU aktif')
    model = nn.DataParallel(model)
model = model.to(device)


DataParallel: 2 GPU aktif


In [14]:


# Loss, Optimizer, Scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
num_epochs = 50
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs, pct_start=0.1
)
scaler = GradScaler(device=device.type)


In [ ]:


# Egitim Dongusu
best_val_loss = float('inf')
print('\n--- RSNA 2023 Egitimi Basliyor ---')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} Train'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} Val'):
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    val_loss = running_loss / len(val_loader.dataset)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1:02d}/{num_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr_now:.2e}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, '/kaggle/working/rsna2023_best_attention_model.pth')
        print('  => Best model kaydedildi')



--- RSNA 2023 Egitimi Basliyor ---


Epoch 1/50 Val: 100%|██████████| 55/55 [00:14<00:00,  3.72it/s]


Epoch 01/50 | Train: 0.3462 | Val: 0.2419 | LR: 2.64e-05
  => Best model kaydedildi


Epoch 2/50 Val: 100%|██████████| 55/55 [00:13<00:00,  3.97it/s]


Epoch 02/50 | Train: 0.2199 | Val: 0.2285 | LR: 7.44e-05
  => Best model kaydedildi


...

Epoch 14/50 Val: 100%|██████████| 55/55 [00:13<00:00,  4.00it/s]


Epoch 14/50 | Train: 0.2119 | Val: 0.2191 | LR: 1.81e-04


Epoch 15/50 Val: 100%|██████████| 55/55 [00:13<00:00,  3.99it/s]


Epoch 15/50 | Train: 0.2110 | Val: 0.2210 | LR: 1.77e-04


Epoch 16/50 Train:   9%|▉         | 20/218 [00:17<02:47,  1.18it/s]